<h1 style="padding-top: 25px;padding-bottom: 25px;text-align: left; padding-left: 10px; background-color: #DDDDDD;
    color: black;"> <img style="float: left; padding-right: 10px;" src="https://raw.githubusercontent.com/Harvard-IACS/2018-CS109A/master/content/styles/iacs.png" height="50px"> <a href='https://harvard-iacs.github.io/2025-AC215/' target='_blank'><strong><font color="#A41034">AC215/E115: MLOps & LLMOps: Production AI Systems</font></strong></a></h1>

# **<font color="#A41034">Tutorial -  Agent Harness:  Formaggio Store</font>**

**Harvard University**<br/>
**Fall 2026**<br/>
**Instructor:** Pavlos Protopapas / Shivas Jayaram<br/>


<hr style="height:2pt">

## 📝 Make a Copy to Edit

This notebook is **view-only**. To edit it, follow these steps:

1. Click **File** > **Save a copy in Drive**.
2. Your own editable copy will open in a new tab.

Now you can modify and run the code freely!

# **Building an Agent Harness**


<center><img src="https://drive.google.com/uc?export=view&id=1oCldVicFepwdYbPHcD8Ui4VeyQVJYgut" height="400"><center>



In this notebook, we will bring together the concepts from the lecture: **context engineering**, **agent harnesses**, **skills**, **agent systems**, and **evaluation**. You will explore a cheese-shop assistant, add controlled workflows and human review, and inspect the evidence of what the system actually did.

As we build the harness we will review the code and compare behaviors to explain which parts are controlled by the model and which are enforced by the application.


# **Learning Objectives**

Help a customer plan a cheese tasting while preserving their constraints and controlling actions.

| Act | Question |
|---|---|
| 1 — Tools | What can an agent with tool use propose? |
| 2 — Context | What useful information did we add, and did the agent use it appropriately? |
| 3 — Enforcement | Will the workflow reject an invalid proposal before checkout and require any necessary approval? |
| 4 — Harness | Can the agent plan a sourcing task, handle untrusted vendor text, and leave the email decision to a human? |
| 5 — Skills | Does the agent load the right skill for the task, while still respecting the harness’s checks? |
| 6 — Agent Systems | Can the assistant use the governed ordering workflow, then turn its verified result into a complete tasting plan? |

Use the recorded results to answer each question. Orders are simulated, emails are saved locally, and the PA restriction is a fictional classroom policy.


# **Understanding Agent Harness**

**What Is an Agent Harness?**<br>
---
A model on its own is just a function from text to text. It can read a prompt and produce a completion — and that is all it can do. It cannot call an API, read a file, remember yesterday, or check whether its own answer was correct.

An **agent harness** is the software wrapped around the model that turns that function into something that can actually do work:

> *"The software scaffolding around a model: the loop, tools, context management, and guardrails that turn raw intelligence into a working agent."* — Anthropic

**What we'll build**

Frameworks (SDKs) usually hide the harness. In this tutorial we do the opposite. We start with the smallest thing that works — an agent with instructions, a few tools, and a customer who wants cheese — and keep that agent fixed.

Then we add each concern (tracing, budgets, context, validation, and later approvals and memory) as a layer around it, and watch what each one adds to the record of what actually happened. By the end, you'll be able to read any agent framework and see the same skeleton underneath.

# Prerequisites

Let's make sure we have everything needed. We'll need to have an OpenAI API key for some parts of this tutorial.

Before we can start using the [OpenAI API](https://openai.com/blog/openai-api), we'll need to sign up for an API key from OpenAI. We can do this by visiting the [OpenAI API Keys](https://platform.openai.com/api-keys) page and creating a new API key.

In [ ]:
# @title Setup Agent Harness Repo
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
# @title Install Dependencies
import tomllib

if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

In [ ]:
# @title Imports
import os
import json
from getpass import getpass
from pathlib import Path

from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient
from openai import AsyncOpenAI

from acts.act1_agent import build_agent
from acts.act2_context import (build_act2, run_act2, preview_context, summarize_context_run,
                               print_context_comparison)
from acts.act3_workflow import build_act3
from acts.act4_harness import build_act4
from acts.act5_skills import build_act5
from acts.act6_composition import build_act6
from cli import (
    print_agent_result,
    print_cart,
    print_workflow_result,
    print_harness_result,
    print_composition_result,
)
from formaggio.config import MODEL, load_json
from formaggio.agents.context import (
    customer_ask, customer_message, instructions, load_scenario,
)
from formaggio.agents.harness import Harness
from formaggio.agents.layers import Trace, Budget, Context, CartCheck
from formaggio.agents.runtime import FUNCTION_LIMITS, MODEL_OPTIONS
from formaggio.agents.tools import build_tools
from formaggio.operations.observability import Recorder
from formaggio.shop.data_models import AgentReply
from formaggio.shop.store import Store
from formaggio.operations.chat_view import show_chat

import pandas as pd
from IPython.display import display

from uuid import uuid4
from formaggio.evaluation.notebook_lab import golden_cases
from formaggio.evaluation.live_evaluation import run_live_suite
from formaggio.operations.evaluation_view import show_evaluation


In [ ]:
# @title Setup OpenAI Key
# Using Google Colab's Secrets feature 🔑
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
import os
os.environ['OPENAI_API_KEY'] = api_key

### Shared settings

The defaults below match the commands in `docs/TUTORIAL_SCENARIOS.md`.
`SCENARIO` selects Act 1; `CONTEXT_SCENARIO` selects Act 2. Later cells name their
scenario explicitly so each cell corresponds to one command in your notes.

Keep `USE_FIXTURES = False` for the live classroom runs. `True` is only for offline
replay in Acts 3–6; in particular, the scripted PA rejection retries before stopping,
whereas the live first run stops at the initial rejection. Acts 1–2 always use a model.
The optional evaluation lab has its own `EVAL_FIXTURES` setting.

Interactive review cells ask you to type `approve` or `decline`. Separate test-decision
cells choose that answer automatically, just like the CLI flags. Run them one at a time.


In [ ]:
# @title Shared settings
SCENARIO = "standard"  # Act 1 baseline.
CONTEXT_SCENARIO = "personalized"  # Same request for both modes in Act 2.
USE_FIXTURES = False  # Scripted responses in Acts 3–6 only; Acts 1–2 are live.
RUN_CONTEXT_EVALUATION = False  # Optional extra live calls at the end of Act 2.
CONTEXT_REPEATS = 3

# Evaluation lab: n different requests, each repeated m times.
EVAL_ACT = 6            # Choose 1–6; start with the complete agent system.
EVAL_N = 3              # First n applicable golden cases shown in the lab.
EVAL_M = 2              # Repeat each case m times (1–20).
EVAL_FIXTURES = True    # False = live model evaluation, with API usage.
EVAL_CONTEXT = "enriched"  # Act 2 only: "basic" or "enriched".

DB_PATH = Path("outputs/colab.sqlite")
OUTPUT_ROOT = Path("outputs/colab_artifacts")
MEMORY_PATH = Path("outputs/colab_memory.sqlite")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def ask_for_decision():
    while True:
        decision = input("Type approve or decline: ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision
        print("Please enter approve or decline.")


async def review_order(ticket):
    print("\n👤 Manager review")
    print(customer_ask(ticket.request))
    print_cart(ticket.report)
    return ask_for_decision()


async def review_email(review):
    print("\n👤 Review this email draft")
    print(f"To: {review.draft.recipient}")
    print(f"Subject: {review.draft.subject}\n")
    print(review.draft.body)
    return ask_for_decision()


async def approve_review(review):
    print("TEST review decision: approve")
    return "approve"


async def decline_review(review):
    print("TEST review decision: decline")
    return "decline"


print(f"Model: {MODEL}")
print("Acts 3–6:", "scripted responses" if USE_FIXTURES else "live model")


# **Formaggio Store**

In this section we look at the mock data for this demo

In [ ]:
# @title Cheese Catalog

pd.set_option("display.width", 140)
catalog = pd.DataFrame(load_json("catalog.json"))
catalog["price_per_100g"] = catalog["cents_per_100g"].map(lambda c: f"${c / 100:.2f}")
catalog["allergens"] = catalog["allergens"].str.join(", ")
catalog[["product_id", "name", "country", "style", "funk", "raw_milk", "price_per_100g", "stock_g", "allergens"]]

In [ ]:
# @title Store Policies
policy = load_json("policies.json")
rules = {
    "Cheeses per tasting":            f'{policy["min_products"]}–{policy["max_products"]} distinct',
    "Grams per guest":                f'{policy["min_grams_per_guest"]}–{policy["max_grams_per_guest"]} g',
    "Minimum per cheese":             f'{policy["min_product_grams"]} g, in {policy["quantity_increment_g"]} g steps',
    "Manager approval above":         f'${policy["manager_threshold_cents"] / 100:.0f} cheese subtotal',
    "Raw-milk shipping blocked to":   ", ".join(policy["raw_milk_blocked_states"]),
    "Revisions allowed":              policy["max_revisions"],
    "Approved vendor recipients":     ", ".join(policy["allowed_vendor_recipients"]),
}
pd.Series(rules, name="value").to_frame()

In [ ]:
# @title Pairings
pairings = pd.DataFrame(load_json("pairings.json"))
pairings["styles"] = pairings["styles"].str.join(", ")
pairings["allergens"] = pairings["allergens"].map(lambda a: ", ".join(a) or "—")
pairings[["name", "kind", "styles", "allergens"]]

In [ ]:
# @title Customers
pd.DataFrame(load_json("customers.json"))

In [ ]:
# @title The Customer Request

request = load_scenario(SCENARIO)
display(customer_ask(request))
print()
print("Structured fields the application actually trusts:")
print(json.dumps(request.model_dump(mode="json"), indent=2))

In [ ]:
# @title Apply the selected request to the shelf

shelf_store = Store()
allergy_tags, allergy_issues = shelf_store.allergy_tags(request)
raw = pd.DataFrame(load_json("catalog.json"))
shipping_label = f"raw milk blocked → {request.state or 'destination unknown'}"
raw[shipping_label] = raw["raw_milk"] & (
    (request.state or "").strip().upper() in shelf_store.policy.raw_milk_blocked_states
)
raw["allergen conflict"] = raw["allergens"].map(lambda tags: bool(allergy_tags.intersection(tags)))
raw["insufficient stock"] = raw["stock_g"] < shelf_store.policy.min_product_grams
flags = [shipping_label, "allergen conflict", "insufficient stock"]
raw["passes product filters"] = ~raw[flags].any(axis=1)
requirements = []
for country in request.required_countries:
    label = f"from {country}"
    raw[label] = raw["country"].str.casefold() == country.casefold()
    requirements.append(label)
if request.required_min_funk is not None:
    label = f"funk ≥ {request.required_min_funk}"
    raw[label] = raw["funk"] >= request.required_min_funk
    requirements.append(label)

view = raw[["product_id", "country", "funk", "stock_g", *flags,
            "passes product filters", *requirements]]
display(view.style
    .map(lambda value: "background-color: #fde2e2" if value else "color: #888", subset=flags)
    .map(lambda value: "background-color: #dff5df" if value else "color: #888",
         subset=["passes product filters", *requirements]))
print("Country and funk requirements may be met by different cheeses.")
print("Passing these product filters does not validate the full cart or resolve missing request details.")
for issue in allergy_issues:
    print("Needs clarification:", issue.detail)


# **The Agent harness**

We now have a shop, its rules, and a customer with a request.

In this section we build the agent that serves that customer, and then wrap it in a harness
one layer at a time.

We keep the shop domain and constraints consistent while changing how the system organizes reasoning, tools, and control.

| Act | We add | So that the system can… |
|---|---|---|
| **1** | The agent, then four harness layers | Propose a cart and leave evidence of what it did |
| **2** | A richer context layer | See the right facts before it starts |
| **3** | A workflow that calls the agent as one step | Validate, revise, pause for a human, and place an order safely |
| **4** | Task list, memory, compaction, and approval | Plan longer work without losing the constraints or overstepping |
| **5** | Skills | Load procedures on demand, without gaining any new permissions |
| **6** | Composition: a harness that calls the Act 3 workflow | Combine flexible planning with controlled execution |

> As a rule of thumb: **the model proposes; the harness decides.**
Every time you see a result, ask which of the two produced it.


## Act 1 — Tools

**Question: What can an agent with tool use propose?**

The model uses tools to look up cheeses, check stock, calculate prices and propose a cart.
First inspect the bare agent, then run it with the tracing, budget, context and cart-check layers.
Neither version can place an order.


### Optional code walkthrough — the bare agent

This extra run shows the model and tool loop before adding the harness.


In [ ]:
# @title Agent

store = Store()
request = load_scenario(SCENARIO)
print("👤 Customer ask:\n" + customer_ask(request))

async with AsyncOpenAI(timeout=45, max_retries=0) as api:
    client = OpenAIChatClient(
        model=MODEL,
        async_client=api,
        function_invocation_configuration=FUNCTION_LIMITS,
    )
    agent = Agent(
        client=client,
        name="FormaggioAssistant",
        instructions=instructions(store),
        tools=build_tools(store, request, None, None),
        default_options={**MODEL_OPTIONS, "response_format": AgentReply},
    )
    bare_response = await agent.run(customer_message(request))

print("\n💬 Agent's response:\n" + bare_response.value.message)


### Add checks around the agent

The harness runs the same agent definition with four named layers:

- **Trace:** show and record what happened.
- **Budget:** limit model calls, tool calls, and runtime.
- **Context:** choose the extra information supplied to the agent.
- **CartCheck:** independently check the final proposed cart.

`build_agent` contains the agent definition shown above, with hooks for these layers. Calling `.run(...)` starts a fresh run using the chosen scenario.

The cart check reports problems; it does not repair the cart or place an order.

### Run Act 1

CLI equivalent: `python cli.py --act 1 --scenario standard`

Look for tool calls, the final proposal and the independent cart check.


In [ ]:
# @title Agent + Harness

act1 = (
    Harness(build_agent, model=MODEL, scenario=SCENARIO)
    .add(Trace())
    .add(Budget(model_calls=8, tool_calls=20, seconds=120))
    .add(Context("basic"))
    .add(CartCheck())
)

with Recorder(DB_PATH) as recorder:
    act1_result = await act1.run(recorder, progress=print)

print_agent_result(act1_result)

In [ ]:
# Response View
show_chat(DB_PATH, act1_result["run_id"])

## Act 2 — Context

**Question: What useful information did we add, and did the agent use it appropriately?**

The current request stays the same. Enriched context adds the customer's saved preference
for mild cheeses and nonalcoholic pairings, plus a filtered catalog. Compare the choices,
not just the number of calls. Our classroom definition of mild is funk level 0–2.


In [ ]:
context_preview = preview_context(CONTEXT_SCENARIO)

print("📚 What changes before the first model call?")
print("Basic: the same shop rules, customer request, and tools; no extra context sources.")
for item in context_preview["enriched"]["sources"]:
    print(f"Enriched adds {item['source']}: {item['reason']}")

print("\nSaved preferences:", context_preview["enriched"]["sources"][-1]["content"].get("preferences", []))

print("\n🔎 Excluded from the enriched starting list:")
reason_labels = {
    "fictional_shipping_policy": "raw milk: blocked by the fictional destination policy",
    "allergen_conflict": "conflicts with the confirmed allergies",
    "insufficient_stock_for_minimum": "not enough stock for the minimum serving",
}
for product in context_preview["enriched"]["excluded_products"]:
    reasons = "; ".join(reason_labels.get(reason, reason) for reason in product["reasons"])
    print(f"- {product['product_id']}: {reasons}")


### Run basic context

CLI equivalent: `python cli.py --act 2 --scenario personalized --context basic`

Inspect the customer preferences supplied and the cheeses actually chosen.


In [ ]:
act2_basic = build_act2(scenario=CONTEXT_SCENARIO, mode="basic", model=MODEL)
with Recorder(DB_PATH) as recorder:
    act2_basic_result = await act2_basic.run(recorder, progress=print)
print_agent_result(act2_basic_result)
show_chat(DB_PATH, act2_basic_result["run_id"], title="Act 2 · basic")


### Run enriched context

CLI equivalent: `python cli.py --act 2 --scenario personalized --context enriched`

Inspect the customer preferences supplied and the cheeses actually chosen.


In [ ]:
act2_enriched = build_act2(scenario=CONTEXT_SCENARIO, mode="enriched", model=MODEL)
with Recorder(DB_PATH) as recorder:
    act2_enriched_result = await act2_enriched.run(recorder, progress=print)
print_agent_result(act2_enriched_result)
show_chat(DB_PATH, act2_enriched_result["run_id"], title="Act 2 · enriched")


### Run a fresh comparison

CLI equivalent: `python cli.py --act 2 --compare`

This makes two new runs, just like the CLI comparison. It does not reuse the answers above.
Read the mild-cheese percentages, cart checks and token usage. A tie or regression is
an honest result; read pairing recommendations yourself too.


In [ ]:
with Recorder(DB_PATH) as recorder:
    comparison_results = await run_act2(
        recorder, scenario=CONTEXT_SCENARIO, mode="both", model=MODEL,
        on_result=print_agent_result,
    )
    context_events = {result["run_id"]: recorder.timeline(result["run_id"])
                      for result in comparison_results}
context_results = {result["context"]["mode"]: result for result in comparison_results}
print_context_comparison(comparison_results, context_events)
for mode, result in context_results.items():
    show_chat(DB_PATH, result["run_id"], title=f"Act 2 · comparison · {mode}")


### Optional: repeat the context comparison

One run is an example, not a reliable comparison. Set `RUN_CONTEXT_EVALUATION = True`
in the shared settings, then run the next cell to try each mode `CONTEXT_REPEATS` times.
It makes **additional live model calls**, including when `USE_FIXTURES = True`.
Leave the switch off to skip it. Every trial starts with fresh shop inventory.

Compare the fraction of carts that pass, the average number of calls, and input tokens.
If both modes perform equally well, the experiment has not shown a benefit from adding
context for this task. A few trials are still a small sample; record mixed or failed
results rather than choosing only the best examples.


In [ ]:
context_trials = []
if RUN_CONTEXT_EVALUATION:
    if type(CONTEXT_REPEATS) is not int or CONTEXT_REPEATS < 2:
        raise ValueError("Use at least two repeats for a comparison.")
    for repetition in range(CONTEXT_REPEATS):
        for mode in ("basic", "enriched"):
            lesson = build_act2(scenario=CONTEXT_SCENARIO, mode=mode, model=MODEL)
            try:
                with Recorder(DB_PATH) as recorder:
                    trial = await lesson.run(recorder)
                    row = summarize_context_run(trial, recorder.timeline(trial["run_id"]))
                row["error"] = None
            except Exception as error:
                row = {"mode": mode, "cart_check": "error", "error": str(error)}
            row["repetition"] = repetition + 1
            context_trials.append(row)
            print(f"Trial {repetition + 1} · {mode}: {row['cart_check']}")
            if row["error"]:
                print("  Error:", row["error"])

    print("\n📊 Repeated comparison")
    for mode in ("basic", "enriched"):
        rows = [row for row in context_trials if row["mode"] == mode]
        completed = [row for row in rows if not row["error"]]
        passed = sum(row["cart_check"] == "passed" for row in rows)
        print(f"{mode}: {passed}/{len(rows)} carts passed; {len(rows) - len(completed)} run errors")
        personalized = sum(row.get("preference_check") == "pass" for row in rows)
        print(f"  Applicable context checks passed: {personalized}/{len(rows)} (basic receives no saved preference)")
        for metric in ("model_calls", "tool_calls", "input_tokens", "output_tokens"):
            measured = [row[metric] for row in completed if row[metric] is not None]
            average = f"{sum(measured) / len(measured):.1f}" if measured else "unavailable"
            print(f"  Average {metric}: {average} ({len(measured)}/{len(rows)} trials measured)")
else:
    print("Optional repeated comparison skipped. Enable it in the shared settings when ready.")


## Act 3 — Enforcement

**Question: Will the workflow reject an invalid proposal before checkout and require any necessary approval?**

Both runs begin with the customer's cart: Comté 350 g, Bûcheron 350 g and Époisses 300 g
for PA. The live model records this input; the harness checks it. The first run stops
at rejection. The second lets the model choose a replacement after receiving feedback.

Watch the raw-milk flag, the rejection, the removed/added cheeses and the receipt.
A failed revision must never reach checkout. These two runs demonstrate shipping;
manager approval is exercised in Act 6.


### Check only

CLI equivalent: `python cli.py --act 3`

Expect Comté to be rejected and no receipt.


In [ ]:
act3_before = build_act3(scenario="pa-shipping-blocked", model=MODEL, fixture=USE_FIXTURES)
with Recorder(DB_PATH) as recorder:
    act3_before_result = await act3_before.run(recorder, manager=review_order, progress=print)
print_workflow_result(act3_before_result)
show_chat(DB_PATH, act3_before_result["run_id"], title="Act 3 · before")


### Allow revision

CLI equivalent: `python cli.py --act 3 --scenario pa-shipping`

Expect the same rejection, followed by a model revision. Only a valid cart gets a receipt.


In [ ]:
act3_after = build_act3(scenario="pa-shipping", model=MODEL, fixture=USE_FIXTURES)
with Recorder(DB_PATH) as recorder:
    act3_after_result = await act3_after.run(recorder, manager=review_order, progress=print)
print_workflow_result(act3_after_result)
show_chat(DB_PATH, act3_after_result["run_id"], title="Act 3 · after")


## Act 4 — Harness

**Question: Can the agent plan a sourcing task, handle untrusted vendor text, and leave the email decision to a human?**

The agent checks a stock shortage and prepares a supplier email. Look for the task list,
verified shortfalls and the review step. Approval saves a local HTML email; refusal saves none.
No message is transmitted. Each cell below starts a new run.


### Human review

CLI equivalent: `python cli.py --act 4`

Review the draft when prompted.


In [ ]:
act4_interactive = build_act4(
    scenario="event-shortage", model=MODEL, fixture=USE_FIXTURES,
    memory_path=MEMORY_PATH, output_root=OUTPUT_ROOT,
    decision_source="human",
)
with Recorder(DB_PATH) as recorder:
    act4_interactive_result = await act4_interactive.run(recorder, reviewer=review_email, progress=print)
print_harness_result(act4_interactive_result)
show_chat(DB_PATH, act4_interactive_result["run_id"], title="Act 4 · interactive")


### Test decision: approve

CLI equivalent: `python cli.py --act 4 --email-decision approve`

The host supplies this test decision; the model cannot approve its own action.


In [ ]:
act4_approve = build_act4(
    scenario="event-shortage", model=MODEL, fixture=USE_FIXTURES,
    memory_path=MEMORY_PATH, output_root=OUTPUT_ROOT,
    decision_source="test_option",
)
with Recorder(DB_PATH) as recorder:
    act4_approve_result = await act4_approve.run(recorder, reviewer=approve_review, progress=print)
print_harness_result(act4_approve_result)
show_chat(DB_PATH, act4_approve_result["run_id"], title="Act 4 · approve")


### Test decision: decline

CLI equivalent: `python cli.py --act 4 --email-decision decline`

The host supplies this test decision; the model cannot approve its own action.


In [ ]:
act4_decline = build_act4(
    scenario="event-shortage", model=MODEL, fixture=USE_FIXTURES,
    memory_path=MEMORY_PATH, output_root=OUTPUT_ROOT,
    decision_source="test_option",
)
with Recorder(DB_PATH) as recorder:
    act4_decline_result = await act4_decline.run(recorder, reviewer=decline_review, progress=print)
print_harness_result(act4_decline_result)
show_chat(DB_PATH, act4_decline_result["run_id"], title="Act 4 · decline")


To exercise malicious vendor text as a separate experiment, add
`document="malicious"` to one of the Act 4 builders above and rerun it. Look for the
suspicious instructions being blocked while customer constraints remain protected.
The clean-document runs above alone do not demonstrate rejection of an attack.


## Act 5 — Skills

**Question: Does the agent load the right skill for the task, while still respecting the harness’s checks?**

A skill supplies reusable instructions and supporting files. Compare the three tasks
below using `Skills loaded` and `Skill resources`. Skills do not bypass approval or
change customer constraints.


### Sourcing task

CLI equivalent: `python cli.py --act 5`

Load the outreach skill and email template. Review the email when prompted.


In [ ]:
act5_sourcing = build_act5(
    scenario="event-shortage", model=MODEL, fixture=USE_FIXTURES,
    memory_path=MEMORY_PATH, output_root=OUTPUT_ROOT,
)
with Recorder(DB_PATH) as recorder:
    act5_sourcing_result = await act5_sourcing.run(recorder, reviewer=review_email, progress=print)
print_harness_result(act5_sourcing_result)
show_chat(DB_PATH, act5_sourcing_result["run_id"], title="Act 5 · event-shortage")


### Stock question

CLI equivalent: `python cli.py --act 5 --scenario stock-question`

Answer from inventory without loading an unnecessary skill.


In [ ]:
act5_stock = build_act5(
    scenario="stock-question", model=MODEL, fixture=USE_FIXTURES,
    memory_path=MEMORY_PATH, output_root=OUTPUT_ROOT,
)
with Recorder(DB_PATH) as recorder:
    act5_stock_result = await act5_stock.run(recorder, reviewer=review_email, progress=print)
print_harness_result(act5_stock_result)
show_chat(DB_PATH, act5_stock_result["run_id"], title="Act 5 · stock-question")


### Tasting plan

CLI equivalent: `python cli.py --act 5 --scenario tasting-plan`

Load the tasting-planning skill and deliver a plan matching the checked menu.


In [ ]:
act5_tasting = build_act5(
    scenario="tasting-plan", model=MODEL, fixture=USE_FIXTURES,
    memory_path=MEMORY_PATH, output_root=OUTPUT_ROOT,
)
with Recorder(DB_PATH) as recorder:
    act5_tasting_result = await act5_tasting.run(recorder, reviewer=review_email, progress=print)
print_harness_result(act5_tasting_result)
show_chat(DB_PATH, act5_tasting_result["run_id"], title="Act 5 · tasting-plan")


## Act 6 — Agent Systems

**Question: Can the assistant use the governed ordering workflow, then turn its verified result into a complete tasting plan?**

The outer assistant calls the ordering workflow with `start_order`, then uses a skill to
explain the accepted menu. Check the authoritative order result and the tasting plan
separately. Cheeses, quantities and prices must agree with the receipt.

The manager scenario requires review only if the proposed subtotal exceeds $200.
A live model may choose a cheaper cart. Check that review actually happened before
claiming approval or refusal was tested. Each cell starts a fresh order.


### Complete order and tasting plan

CLI equivalent: `python cli.py --act 6`

Compare outer model calls with workflow model calls, and inspect the delivery check.


In [ ]:
act6 = build_act6(
    scenario="standard",
    model=MODEL,
    fixture=USE_FIXTURES,
)

with Recorder(DB_PATH) as recorder:
    act6_result = await act6.run(
        recorder,
        manager=review_order,
        progress=print,
    )
    act6_events = recorder.timeline(act6_result["run_id"])

print_composition_result(act6_result)

print("\n🔍 Was the requested tasting plan delivered?")
delivery_checks = [
    event["payload"] for event in act6_events
    if event["event_type"] == "delivery.checked"
]
if delivery_checks:
    for check in delivery_checks:
        print("Ready:", check["ready"])
        for error in check.get("errors", []):
            print("-", error)
else:
    print("No delivery check was recorded. Inspect the run status above.")

In [ ]:
show_chat(DB_PATH, act6_result["run_id"])

### Human manager review

CLI equivalent: `python cli.py --act 6 --scenario manager-approval`

Inspect the cart before deciding.


In [ ]:
act6_interactive = build_act6(
    scenario="manager-approval", model=MODEL, fixture=USE_FIXTURES,
    decision_source="human",
)
with Recorder(DB_PATH) as recorder:
    act6_interactive_result = await act6_interactive.run(recorder, manager=review_order, progress=print)
print_composition_result(act6_interactive_result)
show_chat(DB_PATH, act6_interactive_result["run_id"], title="Act 6 · manager · interactive")


### Test manager decision: approve

CLI equivalent: `python cli.py --act 6 --scenario manager-approval --manager-decision approve`

A reviewed-and-declined order must have no receipt. Approval still requires the cart to pass final validation.


In [ ]:
act6_approve = build_act6(
    scenario="manager-approval", model=MODEL, fixture=USE_FIXTURES,
    decision_source="test_option",
)
with Recorder(DB_PATH) as recorder:
    act6_approve_result = await act6_approve.run(recorder, manager=approve_review, progress=print)
print_composition_result(act6_approve_result)
show_chat(DB_PATH, act6_approve_result["run_id"], title="Act 6 · manager · approve")


### Test manager decision: decline

CLI equivalent: `python cli.py --act 6 --scenario manager-approval --manager-decision decline`

A reviewed-and-declined order must have no receipt. Approval still requires the cart to pass final validation.


In [ ]:
act6_decline = build_act6(
    scenario="manager-approval", model=MODEL, fixture=USE_FIXTURES,
    decision_source="test_option",
)
with Recorder(DB_PATH) as recorder:
    act6_decline_result = await act6_decline.run(recorder, manager=decline_review, progress=print)
print_composition_result(act6_decline_result)
show_chat(DB_PATH, act6_decline_result["run_id"], title="Act 6 · manager · decline")


## Evaluation lab — Test the agents across requests and repeated runs

We have watched individual runs. Now ask: **does the system do the right thing across
several requests, and does it keep doing the right thing when we repeat them?**

A **golden case** is a mock request with an expected outcome. We check properties,
such as “ask for missing details” or “place one valid mock order,” rather than requiring
one exact answer or cheese selection. A refusal or clarification can be a success.

Use the shared settings at the top:

- `EVAL_ACT`: which act to evaluate; start with Act 6, which combines the assistant and workflow.
- `EVAL_N`: number of different applicable cases, selected from the list below.
- `EVAL_M`: repetitions of each case. There are **n × m scheduled runs**.
- `EVAL_FIXTURES`: start with `True` to learn the evaluation process using scripted responses.
  Set it to `False` to measure the live model. This setting is independent of `USE_FIXTURES`.
- `EVAL_CONTEXT`: select basic or enriched context when evaluating Act 2.

With the defaults, three requests run twice: **six scripted runs, with no API calls**.
Scripted consistency does not demonstrate model robustness. Live runs make multiple
model calls per request and incur API usage. To study one request repeatedly, use
`EVAL_N = 1` and increase `EVAL_M`.

Every run starts with fresh inventory, session state, and preference memory. Review
choices are explicit test decisions supplied by each case, even in live mode; the
model never acts as its own manager. No real purchases or emails are sent.

### 1. Inspect the golden requests and expected outcomes

The readable requests below summarize the confirmed scenario data. The agents receive
that data through their normal act-specific inputs; exact model messages remain in
the trace. Changing this display text does not change the test. Case definitions live
in `data/agent_evaluation_cases.json`, with request fields in `data/scenarios.json`.


In [ ]:
available_golden_cases = golden_cases(EVAL_ACT)
if type(EVAL_N) is not int or not 1 <= EVAL_N <= len(available_golden_cases):
    raise ValueError(f"For Act {EVAL_ACT}, choose EVAL_N from 1 to {len(available_golden_cases)}.")
if type(EVAL_M) is not int or not 1 <= EVAL_M <= 20:
    raise ValueError("Choose EVAL_M from 1 to 20; use at least 2 to explore repeatability.")
if EVAL_ACT == 2 and EVAL_CONTEXT not in {"basic", "enriched"}:
    raise ValueError("For this lab, choose one context mode: basic or enriched.")

selected_golden_cases = available_golden_cases[:EVAL_N]
for index, case in enumerate(available_golden_cases, 1):
    print(f"\n{'✅ Selected' if index <= EVAL_N else '◻️ Available'} · {index}. {case['case_id']}")
    print("👤 Request:", case["request"])
    print("🎯 Expected:", case["expected"])
    print(f"Review decision if requested: {case['review_decision']}; vendor document: {case['document']}")

print(f"\nScheduled: {EVAL_N} requests × {EVAL_M} repeats = {EVAL_N * EVAL_M} runs")
print("Mode:", "SCRIPTED — tests the evaluation machinery" if EVAL_FIXTURES else "LIVE — measures model behavior")

evaluation_selection = (EVAL_ACT, EVAL_N, EVAL_M, EVAL_CONTEXT)


### 2. Run the selected cases

Each execution gets a new experiment label, so rerunning the cell preserves the previous
batch. The runner uses the real act implementations and the same checks as the CLI.

It records successful runs, failed checks, execution errors, and any missing scheduled
runs. After three consecutive execution errors it stops the batch; missing runs still
count in the pass-rate denominator. Manager-review cases also report whether review
was actually reached—a cheap cart can bypass that branch without testing it.


In [ ]:
# Rerun the selection cell after changing the evaluation settings.
if evaluation_selection != (EVAL_ACT, EVAL_N, EVAL_M, EVAL_CONTEXT):
    raise ValueError("Evaluation settings changed; rerun the golden-case selection cell first.")
evaluation_label = f"colab-act{EVAL_ACT}-{uuid4().hex[:12]}"


def evaluation_progress(message):
    # Full model/tool traces are saved; keep the notebook's progress output short.
    if message.startswith(("FIXTURE", "LIVE", "Case ")):
        print(message)


with Recorder(DB_PATH) as recorder:
    evaluation_report = await run_live_suite(
        recorder,
        evaluation_label,
        act=EVAL_ACT,
        case_ids=[case["case_id"] for case in selected_golden_cases],
        repeats=EVAL_M,
        context=EVAL_CONTEXT if EVAL_ACT == 2 else None,
        model=MODEL,
        fixture=EVAL_FIXTURES,
        output_root=OUTPUT_ROOT,
        progress=evaluation_progress,
    )
    evaluation_export = {
        "report": evaluation_report,
        "traces": {run["run_id"]: recorder.timeline(run["run_id"])
                   for run in evaluation_report["runs"]},
    }

evaluation_json = OUTPUT_ROOT / f"{evaluation_label}.json"
evaluation_json.write_text(json.dumps(evaluation_export, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\nSaved complete results and recorded events: {evaluation_json}")
print(f"Experiment: {evaluation_label} · database: {DB_PATH}")


### 3. Read the scorecard and inspect every run

The view below reads the saved batch; rerunning this display cell makes no model calls.

| Metric | What it tells us |
|---|---|
| **Passed / scheduled** | How often all applicable automated checks passed; errors and missing runs remain in the denominator. |
| **Outcome agreement** | How often the most common completed outcome occurred across the scheduled repeats. Consistent behavior can still be wrong. |
| **Distinct results, responses, and tool paths** | How much the outputs and execution paths vary. Different valid menus or wording are allowed. |
| **Model calls and tokens** | How much model work each run required, including inner and outer agents. Missing measurements are shown as unavailable. |
| **Latency** | Recorded end-to-end run time, including tools and scripted review handling. |
| **Untested paths** | Checks such as manager review that the run never reached. |

Expand any run to inspect **expected vs. observed checks, the final output, and the full
sequence of recorded events**. Large payload previews may be shortened; the JSON export
contains the complete recorded events. In Colab, use the Files sidebar to download the
printed JSON file or the SQLite database for later inspection.

There is no automatic prose-quality score. Read at least one passing and one failing
run, when available. Check clarity, accurate action status, and whether the response
actually helps the customer. A small batch estimates behavior; it does not prove robustness.


In [ ]:
show_evaluation(DB_PATH, evaluation_report)


### 4. Explain what you learned

1. Which request passed least often? Expand its runs and identify the failing check.
2. Did the same request produce different valid menus, different outcomes, or errors?
3. Which case used the most tokens or took longest? What does its trace explain?
4. Did a manager-review case actually exercise approval or decline?
5. Record the mode, model, case IDs, number of repeats, and experiment label with your conclusion.

Start with scripted runs to understand the scorecard, then switch to live mode for a
small batch. To compare two acts, use case IDs that apply to both and account for their
different responsibilities: Act 1 proposes a cart; Act 6 can place a mock order and deliver
a plan. Matching wording is not the definition of success.


## Reflection — Answer the six tutorial questions

Return to each act's question and point to a recorded result that answers it.

- Which choices came from the model, and which checks came from application code?
- Did enriched context improve personalization in your comparison?
- Which cheese was rejected for PA, and what replaced it?
- Did the email and order decisions actually reach human review?
- Which tasks needed a skill, and did the final plan match the checked menu?

A single good result is an example. Use the evaluation lab to examine repeated runs,
and keep failures and untested approval paths in your conclusions.
